# 10 · Tucker decomposition on real data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Descomposición de Tucker con datos reales** — PCA generalizado a todos los ejes, sobre un tensor real de viajes en taxi de Nueva York.

PCA generalized to every axis, on a real tensor of New York taxi trips.

## What you will be able to do

- Build a genuine order-3 tensor out of a flat table of real trips.
- Compute a Tucker decomposition by HOSVD, using only unfolding, SVD and einsum.
- Contract three axes at once with a single `einsum` string.
- Measure reconstruction error against compression ratio.
- Read a factor matrix and recognise a real pattern the decomposition found by itself.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd

TAXIS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
taxis = pd.read_csv(TAXIS)

def unfold(T, axis):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

print(taxis.shape)                       # (6433, 14) — 6,433 real NYC taxi trips

## The theory

> 🇪🇸 PCA comprime una **matriz**: dos ejes. La descomposición de Tucker
> generaliza PCA a un tensor de cualquier orden: una **matriz de factores por
> eje**, más un **tensor núcleo** pequeño.

PCA compresses a **matrix** — two axes. Real data often has more. **Tucker
decomposition** generalizes PCA to a tensor of any order: one **factor matrix
per axis**, plus a small **core tensor** describing how the factors combine.

The way to compute it, called **HOSVD**, uses only tools you already have:

1. **Unfold** the tensor along each axis (section 01).
2. Run **SVD** on each unfolding; keep the top components. These are the factor
   matrices.
3. **Contract** the original tensor against all factor matrices to get the core
   (section 06).

The related **CP decomposition** instead writes the tensor as a sum of simple
rank-1 pieces. Tucker is usually more accurate at the same size; CP is often
easier to interpret.

## Our real tensor

From 6,433 real New York taxi trips we build a genuine order-3 tensor:
**pickup borough × dropoff borough × hour of day.**

> 🇪🇸 Un tensor real de orden 3: barrio de origen × barrio de destino × hora.

In [ ]:
taxis['hour'] = pd.to_datetime(taxis['pickup']).dt.hour
sub = taxis.dropna(subset=['pickup_borough', 'dropoff_borough'])
pb = sorted(sub['pickup_borough'].unique())
db = sorted(sub['dropoff_borough'].unique())

T = np.zeros((len(pb), len(db), 24))
for (p, d, h), v in sub.groupby(['pickup_borough', 'dropoff_borough', 'hour']).size().items():
    T[pb.index(p), db.index(d), h] = v

print(T.shape, pb, db)

## Exercise 1 — read the tensor before you decompose it

> 🇪🇸 Entiende el tensor antes de descomponerlo.

In [ ]:
# TODO 1: Print T.shape and T.sum(). What does the entry T[i, j, k] mean?

# TODO 2: Which hour has the most trips overall? (Sum over the first two axes.)

# TODO 3: Unfold T along each axis and print the three shapes. Confirm the total
#         number of entries is the same each time — unfolding loses nothing.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(T.shape, T.sum())
# T[i, j, k] = how many trips started in borough pb[i], ended in borough db[j],
# and were picked up during hour k.

by_hour = T.sum(axis=(0, 1))
print(by_hour.argmax())                    # 18 — evening rush hour

for ax in range(3):
    M = unfold(T, ax)
    print(ax, M.shape, M.size == T.size)   # True every time

## Exercise 2 — HOSVD, in two einsum calls

> 🇪🇸 HOSVD en dos llamadas a einsum.

Look at the einsum strings you are about to write: `'ijk,ia,jb,kc->abc'`
contracts three axes in one expression. **That is why einsum came first.**

In [ ]:
# TODO 4: Run SVD on each unfolding, keep the top (2, 2, 3) components, and
#         build the core tensor with ONE einsum call.

# TODO 5: Reconstruct T from the core and factors, again with one einsum.
#         Compute the relative error and the compression ratio.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
Us = [np.linalg.svd(unfold(T, ax), full_matrices=False)[0] for ax in range(3)]
r = (2, 2, 3)
Us = [Us[i][:, :r[i]] for i in range(3)]
print([u.shape for u in Us])

core  = np.einsum('ijk,ia,jb,kc->abc', T, Us[0], Us[1], Us[2])   # (2, 2, 3)
recon = np.einsum('abc,ia,jb,kc->ijk', core, Us[0], Us[1], Us[2])

error = np.linalg.norm(T - recon) / np.linalg.norm(T)            # 0.067
ratio = T.size / (core.size + sum(u.size for u in Us))           # 4.71
print(core.shape, round(error, 3), round(ratio, 2))

## Exercise 3 — what did it find?

> 🇪🇸 ¿Qué encontró la descomposición por sí sola?

This is the important one.

In [ ]:
# TODO 6: Look at the first column of the hour factor matrix. At which hour is
#         it largest? Does that match what you found in TODO 2?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
hour_factor = Us[2]                        # (24, 3) — one row per hour
peak = np.abs(hour_factor[:, 0]).argmax()
print(peak)                                # 18

print(T.sum(axis=(0, 1)).argmax())         # 18 — the same hour, from raw counts

# THE DECOMPOSITION DISCOVERED EVENING RUSH HOUR BY ITSELF. Nobody told it about
# time, traffic or commuting; it found the dominant pattern along that axis
# because that is what a decomposition does.
#
# (Take the absolute value: singular vectors are only defined up to sign, so the
# strongest component may come out negative.)

## What just happened

**4.7× fewer numbers, 6.7% error.** But the important part is TODO 6. The
strongest pattern in the hour factor peaks at **hour 18** — and that is also the
busiest hour in the raw data. The decomposition found rush hour on its own.

**Where this is used.** In tech, Tucker and CP compress the large weight tensors
inside neural networks so models run on phones instead of servers. In biotech,
applied to data such as (genes × samples × conditions), they find structure
ordinary PCA cannot reach, because **PCA can only ever see two axes**.

For real projects use [`tensorly`](https://tensorly.org), which implements both
properly. Take-home C in section 11 compares CP against what you just built.

---

## Time for Kahoot 🎯

**Kahoot 3 — Convolution & Tensor Decompositions** · 6 questions, about 5 minutes.

> 🇪🇸 **Convolución y descomposiciones tensoriales** — 6 preguntas, unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-3)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_3_convolution_decompositions.xlsx)

Next up: **11 · Wrap-up and take-homes** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)